In [2]:
import os, sys

# Go up THREE levels (project root directory)
project_root = os.path.dirname(os.path.dirname(os.getcwd()))
project_root
# Append the new path to sys.path
if project_root not in sys.path:
    sys.path.append(project_root)
    print("Project root added to sys.path")
else:
    print("Project root already in sys.path")

Project root added to sys.path


In [3]:
import numpy as np
import trimesh
from utils import gro_processing as gp 

# --- CONFIG ---
gro_path = "../../data/npt-HK4.gro"             # <- change if needed
sphere_radius_scale = 0.3             # balls (atoms): 2.0 × van der Waals radius
bond_radius = 0.15                     # sticks (bonds): cylinder radius in Å
sphere_subdiv = 2                         # atom sphere detail (2 is moderate)

atoms, title, num_atoms, box_dimensions = gp.read_gro(gro_path)
atoms[['x','y','z']] = atoms[['x','y','z']].apply(lambda x : x * 10) # convert to Å

# # --- Minimal GRO parser (coords in nm -> convert to Å) ---
# def parse_gro(path):
#     with open(path, "r") as f:
#         _title = f.readline()
#         n = int(f.readline().strip())
#         n = 84
#         atoms = []
#         for _ in range(n):
#             line = f.readline()
#             x = float(line[20:28]) * 10.0
#             y = float(line[28:36]) * 10.0
#             z = float(line[36:44]) * 10.0
#             name = line[10:15].strip()
#             atoms.append((name, np.array([x, y, z], dtype=float)))
#         _ = f.readline()  # box
#     return atoms

# atoms = parse_gro(gro_path)

# # --- Element inference ---
def infer_element(atomname):
    # common water aliases
    if atomname in ("OW", "HW", "HW1", "HW2"): return "O" if atomname=="OW" else "H"
    # simple: first letter, capitalize second if lowercase
    a = ''.join([c for c in atomname if c.isalpha()])
    if not a: return "C"
    if len(a) >= 2 and a[1].islower(): return (a[0]+a[1]).capitalize()
    return a[0].upper()

# --- Radii (Å) ---
vdw = {"H":1.20,"C":1.70,"N":1.55,"O":1.52,"F":1.47,"P":1.80,"S":1.80,"Cl":1.75,"Na":2.27,"K":2.75,"Ca":2.31}
cov = {"H":0.31,"C":0.76,"N":0.71,"O":0.66,"F":0.57,"P":1.07,"S":1.05,"Cl":1.02,"Na":1.66,"K":2.03,"Ca":1.74}

atoms_10 = atoms.loc[756:839] # atom 10
atoms_15 = atoms.loc[1176:1259] # atom 15

def create_trimesh(atoms):
    coords = np.array([p for p in atoms[['x','y','z']].values])
    elements = [infer_element(n) for n in atoms['atom_name'].values]
    vdw_r = np.array([vdw.get(e, 1.70) for e in elements])
    cov_r = np.array([cov.get(e, 0.77) for e in elements])

    # --- Build ball-and-stick meshes ---
    meshes = []

    # balls
    for pos, r in zip(coords, vdw_r * sphere_radius_scale):
        sph = trimesh.creation.icosphere(subdivisions=sphere_subdiv, radius=float(r))
        sph.apply_translation(pos)
        meshes.append(sph)

    # bonds (distance criterion with covalent radii)
    d = np.linalg.norm(coords[None,:,:] - coords[:,None,:], axis=-1)
    thr = 1.2 * (cov_r[:,None] + cov_r[None,:])
    np.fill_diagonal(d, np.inf)
    pairs = np.transpose(np.where(d < thr))
    pairs = pairs[pairs[:,0] < pairs[:,1]]

    for i, j in pairs:
        seg = np.vstack((coords[i], coords[j]))
        cyl = trimesh.creation.cylinder(radius=bond_radius, segment=seg, sections=24)
        meshes.append(cyl)

    molecule = trimesh.util.concatenate(meshes)

    return molecule

molecule_10 = create_trimesh(atoms_10)
molecule_15 = create_trimesh(atoms_15)


# # --- Ray test: choose a ray that shoots toward the molecule's center ---
# bb_min, bb_max = molecule.bounds
# center = (bb_min + bb_max) / 2.0
# origin = bb_min - np.array([10.0, 0.0, 0.0])     # start left of the bbox
# direction = (center - origin); direction /= np.linalg.norm(direction)

# # Intersect (True if any hit)
# hits_any = molecule.ray.intersects_any(
#     ray_origins=origin.reshape(1,3),
#     ray_directions=direction.reshape(1,3)
# )

# print(f"Atoms: {len(atoms)} | Bonds: {len(pairs)}")
# print(f"Ray origin: {origin}, direction: {direction}")
# print("Ray passes through molecule?" , bool(hits_any))


In [4]:
atom_temp = atoms.loc[30996:31079] # 370
molecule_temp = create_trimesh(atom_temp)
molecule_temp.show()

In [5]:
atom_1134 = atoms.loc[95172:95255]
atom_208 = atoms.loc[17388:17471]
atom_370 = atoms.loc[30996:31079]

molecule_370 = create_trimesh(atom_370)
molecule_208 = create_trimesh(atom_208)
molecule_1134 = create_trimesh(atom_1134)

In [7]:
s = trimesh.Scene()
# s.add_geometry(molecule_10)
# s.add_geometry(molecule_15)
s.add_geometry(molecule_1134)
s.add_geometry(molecule_208)
s.add_geometry(molecule_370)


s.show()



In [7]:
from trimesh.collision import CollisionManager

def is_blocked(meshes, idx_A, idx_B, step=0.5):
    """
    Check if molecule A is blocked from seeing molecule B by other molecules.
    meshes: list or array of trimesh.Trimesh objects
    idx_A, idx_B: indices of source and target molecules
    step: step size along the line of sight
    """
    def get_mesh_by_id(meshes, idx):
        for mol_id, mol in meshes:
            if mol_id == idx:
                return mol
        raise ValueError(f"Mesh with id {idx} not found")
    
    def walk_and_check(start_idx, end_idx):
        mol_start = get_mesh_by_id(meshes, start_idx)
        mol_end = get_mesh_by_id(meshes, end_idx)
        
        # centroids
        p_start = mol_start.centroid
        p_end = mol_end.centroid

        # direction vector
        vec = p_end - p_start
        dist = np.linalg.norm(vec)
        direction = vec / dist

        mol_copy = mol_start.copy()

        # build collision manager once (all except start/end)
        manager = CollisionManager()
        for mol_id, mol in meshes:
            if mol_id in (start_idx, end_idx):
                print("not u") #temp
                continue
            try:
                manager.add_object(name=f"mol{mol_id}",mesh=mol)
            except:
                # print(f"fails at {mol_id}")
                pass

        # march along line
        # print(p_end, p_start)
        try:
            num_steps = int(dist / step)
        except:
            print(f"fail at {(start_idx, end_idx)}")
            return True
        
        for _ in range(num_steps):  
            print(f"stepping{_}")
            mol_copy.apply_translation(direction * step)
            if manager.in_collision_single(mol_copy):
                return True  # blocked!
            
        return False # not blocked



            # # check against all other molecules
            # for j, mol in enumerate(meshes):
            #     if j in (start_idx, end_idx):
            #         continue
            #     if not mol_copy.intersection(mol).is_empty:
            #         return True   # blocked!
    return walk_and_check(idx_A,idx_B) or walk_and_check(idx_B,idx_A)


In [5]:
meshes = [[10,molecule_10],[15,molecule_15]]

In [8]:
# Blocking algo
import itertools
edges = []

for pair in itertools.combinations([10,15],2):
    if is_blocked(meshes,pair[0],pair[1]):
            # print(f"{pair}blocked")
            continue
    else:
        edges.append(pair)

edges

# for index, nearby_res in subboxes.values:
#     mask = np.isin(trimesh_arr[:,0], nearby_res)
#     nearby_trimesh_arr = trimesh_arr[mask]
#     for pair in itertools.combinations(nearby_res,2):
#         # print(pair)
#         if is_blocked(nearby_trimesh_arr,pair[0],pair[1]):
#             # print(f"{pair}blocked")
#             continue
#         else:
#             edges.append(pair)

# edges

not u
not u
stepping0
stepping1
stepping2
stepping3
stepping4
stepping5
stepping6
stepping7
stepping8
stepping9
stepping10
stepping11
stepping12
stepping13
stepping14
stepping15
stepping16
stepping17
stepping18
stepping19
stepping20
stepping21
stepping22
stepping23
stepping24
stepping25
stepping26
stepping27
stepping28
stepping29
stepping30
stepping31
stepping32
stepping33
stepping34
stepping35
stepping36
stepping37
stepping38
stepping39
stepping40
stepping41
stepping42
stepping43
stepping44
stepping45
stepping46
stepping47
stepping48
stepping49
stepping50
stepping51
stepping52
stepping53
stepping54
stepping55
stepping56
stepping57
stepping58
stepping59
stepping60
stepping61
stepping62
stepping63
stepping64
stepping65
stepping66
stepping67
stepping68
stepping69
stepping70
stepping71
stepping72
stepping73
stepping74
stepping75
stepping76
stepping77
stepping78
stepping79
stepping80
stepping81
stepping82
stepping83
stepping84
stepping85
stepping86
stepping87
stepping88
stepping89
stepping

[(10, 15)]

In [78]:
meshes = [[1134,molecule_1134],[208,molecule_208],[370,molecule_370]]

In [79]:
edges = []

for pair in itertools.combinations([1134,208,370],2):
    if is_blocked(meshes,pair[0],pair[1]):
            # print(f"{pair}blocked")
            continue
    else:
        edges.append(pair)

edges

not u
not u
stepping0
stepping1
stepping2
stepping3
stepping4
stepping5
stepping6
stepping7
stepping8
stepping9
stepping10
stepping11
stepping12
stepping13
stepping14
stepping15
stepping16
stepping17
stepping18
not u
not u
stepping0
stepping1
stepping2
stepping3
stepping4
not u
not u
stepping0
stepping1
stepping2
stepping3
stepping4
stepping5
stepping6
stepping7
stepping8
stepping9
stepping10
stepping11
stepping12
stepping13
stepping14
stepping15
stepping16
stepping17
stepping18
stepping19
stepping20
stepping21
stepping22
not u
not u
stepping0
stepping1
stepping2
stepping3
stepping4
stepping5
stepping6
stepping7
stepping8
stepping9
stepping10
stepping11
stepping12
stepping13
stepping14
stepping15
stepping16
stepping17
stepping18
stepping19
stepping20
stepping21
stepping22
not u
not u
stepping0
stepping1
stepping2
stepping3
stepping4
stepping5
stepping6


[(1134, 370)]